# Cross-Dataset Phishing Website Detection
## Notebook 2: Feature Alignment and Harmonization

**Objective:** Build an equivalent feature representation between the **PhiUSIIL** and **UCI Phishing Websites** datasets.
- Cross-dataset evaluation requires identical feature space and semantics.
- Uses published rules from Mohammad et al. (2014) to discretize PhiUSIIL continuous values into UCI's `{-1, 0, 1}` domain.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(BASE))

from src.data_utils import project_paths, load_processed
from src.alignment import get_mapping_df, save_mapping_table, build_aligned_datasets

dirs = project_paths(BASE)
PROC_DIR = dirs['processed']
TAB_DIR = dirs['tables']
FIG_DIR = dirs['figures']

### 1. Feature Alignment Mapping Table
Inspect the candidate concepts mapped between datasets, prediction-time availability, and confidence.

In [ ]:
mapping_df = get_mapping_df()
save_mapping_table(TAB_DIR)
display(mapping_df[['concept', 'phi_col', 'uci_col', 'available', 'confidence']])

### 2. Recode Features & Build Aligned Datasets
Apply recoding rules to PhiUSIIL and select equivalent columns from UCI.

In [ ]:
phi_clean, uci_clean, meta = load_processed(PROC_DIR)
phi_al, uci_al = build_aligned_datasets(phi_clean, uci_clean, save_dir=PROC_DIR)

print('Aligned PhiUSIIL shape:', phi_al.shape)
print('Aligned UCI shape     :', uci_al.shape)
assert list(phi_al.columns) == list(uci_al.columns), 'Mismatch in aligned feature columns!'

### 3. Verify Harmonized Feature Distributions
Check that both datasets share identical discrete value levels across all aligned features.

In [ ]:
feat_cols = [c for c in phi_al.columns if c != 'is_phishing']
fig, axes = plt.subplots(len(feat_cols), 2, figsize=(12, 3 * len(feat_cols)))

for idx, col in enumerate(feat_cols):
    sns.countplot(data=phi_al, x=col, hue='is_phishing', ax=axes[idx, 0])
    axes[idx, 0].set_title(f'PhiUSIIL: {col}')
    
    sns.countplot(data=uci_al, x=col, hue='is_phishing', ax=axes[idx, 1])
    axes[idx, 1].set_title(f'UCI: {col}')

plt.tight_layout()
plt.savefig(FIG_DIR / 'aligned_features_distribution.png', dpi=150, bbox_inches='tight')
plt.show()